# CNN Code Illustration

This notebook gives code examples for `02_Introduction_To_CNNs.ipynb`.

It includes:

- Convolution on a tiny image
- Output size calculation
- A small CNN architecture
- Fashion-MNIST training skeleton


In [ ]:
import torch
from torch import nn

torch.manual_seed(42)


## Convolution on a Tiny Image


In [ ]:
image = torch.tensor([
    [0., 0., 0., 0., 0.],
    [0., 1., 1., 1., 0.],
    [0., 1., 1., 1., 0.],
    [0., 1., 1., 1., 0.],
    [0., 0., 0., 0., 0.],
])

kernel = torch.tensor([
    [-1., -1., -1.],
    [ 0.,  0.,  0.],
    [ 1.,  1.,  1.],
])

conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
conv.weight.data = kernel.view(1, 1, 3, 3)

x = image.view(1, 1, 5, 5)
feature_map = conv(x)

print(feature_map.squeeze())


## Output Size Function


In [ ]:
def conv_output_size(input_size, kernel_size, padding=0, stride=1):
    return ((input_size + 2 * padding - kernel_size) // stride) + 1

print(conv_output_size(28, kernel_size=3, padding=1, stride=1))
print(conv_output_size(28, kernel_size=2, padding=0, stride=2))


## A Small CNN


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SmallCNN()
dummy = torch.randn(4, 1, 28, 28)
out = model(dummy)

print(out.shape)


## Optional Fashion-MNIST Training Skeleton


In [ ]:
# This cell needs torchvision and internet access the first time the dataset is downloaded.

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SmallCNN(num_classes=10).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(2):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = loss_fn(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    print(f'Epoch {epoch + 1} | loss = {total_loss / len(train_data):.4f}')


## Evaluation Skeleton


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb).argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

print(f'Test accuracy: {correct / total:.3f}')
